In [1]:
# Clone the repository and navigate into it
!git clone https://github.com/emanuelepietrocometti/anomaly_detection_for_textile_industry.git
%cd anomaly_detection_for_textile_industry

# Install required dependencies
!pip install -r requirements.txt

Cloning into 'anomaly_detection_for_textile_industry'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 178 (delta 120), reused 116 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 325.27 KiB | 1.41 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/anomaly_detection_for_textile_industry
Ignoring colorama: markers 'python_version >= "3.11" and python_version < "3.15" and (platform_system == "Windows" or sys_platform == "win32")' don't match your environment
Ignoring networkx: markers 'python_version == "3.14"' don't match your environment
Ignoring tzdata: markers 'python_version >= "3.11" and python_version < "3.15" and (sys_platform == "win32" or sys_platform == "emscripten")' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 851.8/851.8 kB 51.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 8.5 MB

In [9]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 107.7 MB/s eta 0:00:00


In [3]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Google Drive source paths
drive_source = "/content/drive/MyDrive/Tesi/v4.0.0"
drive_training = "/content/drive/MyDrive/Tesi/dataset_splitted/Training"
drive_validation = "/content/drive/MyDrive/Tesi/dataset_splitted/Validation"

# Colab local destination paths
colab_source = "/content/dataset_source"
colab_training = "/content/dataset_splitted/training"
colab_validation = "/content/dataset_splitted/validation"

os.makedirs(colab_source, exist_ok=True)
os.makedirs(colab_training, exist_ok=True)
os.makedirs(colab_validation, exist_ok=True)


try:
    shutil.copytree(drive_source, colab_source, dirs_exist_ok=True)
    print(f"Copied: {drive_source} -> {colab_source}")
except FileNotFoundError:
    print(f"ERROR: Path '{drive_source}' not found.")


Mounted at /content/drive
Copied: /content/drive/MyDrive/Tesi/v4.0.0 -> /content/dataset_source


In [4]:
import yaml

config_path = "/content/anomaly_detection_for_textile_industry/config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

config["dataset_pipeline"]["paths"]["source_unsplitted_dataset"] = "/content/dataset_source"
config["dataset_pipeline"]["paths"]["source_training"] = "/content/dataset_splitted/training"
config["dataset_pipeline"]["paths"]["source_validation"] = "/content/dataset_splitted/validation"

with open(config_path, "w") as file:
    yaml.dump(config, file)

print("Configuration updated successfully.")

Configuration updated successfully.


In [5]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


False

In [8]:
import sys
import os

PROJECT_ROOT = "/content/anomaly_detection_for_textile_industry"
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

!cd {PROJECT_ROOT} && export PYTHONPATH={SRC_PATH}:{PROJECT_ROOT} && python main.py --create-dataset --baseline patchcore


Starting dataset creation...
--- SPLIT SOURCE IN TRAINING AND VALIDATION DIRECTORY ---
Source dataset splitted into traingin and validation
--- EXTRACTING AND SLICING SOURCE DATA (ZERO WASTE POLICY) ---
Added 356 test only good images from categories ['dust'] directly to AD Test Set.
Source data extracted and sliced

--- GENERATING INVARIANCE FOR NORMAL IMAGES (STRESS TEST) ---
Generating 117 distorted variants to fortify the backbone...

--- TRANSFER LEARNING CLASS BALANCING ---
TL Train Set -> Good (including stress variants): 234 | Defects: 234

--- DATASETS READY FOR TRAINING ---
AD Test Set -> Good: 538 | Defects (Zero Waste): 118

[INFO] No custom weights found for 20260331_210817. Make sure the model supports default weights.

Configuring PATCHCORE...
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is bein